# 99 — Final Pipeline (end-to-end)

This notebook executes the entire pipeline top-to-bottom in one go.  It is what the rubric calls the "interface notebook" and what CI smoke-tests on every push.

It deliberately uses smaller hyperparameter budgets and a smaller data sample than the individual notebooks — the goal here is *demonstrating the pipeline runs*, not winning every metric.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
np.seterr(all="ignore")

ROOT = Path.cwd()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "src").exists():
    sys.path.insert(0, str(ROOT.parent))

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)


In [2]:
import time
t0 = time.time()
print("Stage 1/6: data acquisition")
from src.data.loaders import (
    add_ticket_price, augment_with_aircraft, augment_with_weather,
    load_bts, load_faa_registry, prepare_modelling_frame,
)
from src.data.ec261 import KM_PER_MILE, label_eligible_delay

df = load_bts(fallback="synthetic", n_synthetic=80_000)
df = augment_with_aircraft(df, load_faa_registry())
df = augment_with_weather(df)
df = prepare_modelling_frame(df)
df = add_ticket_price(df, seed=1)
y = label_eligible_delay(df).to_numpy()
print(f"  rows={len(df):,}, base rate={y.mean():.3%}")


Stage 1/6: data acquisition


  rows=78,322, base rate=4.948%


In [3]:
print("Stage 2/6: temporal split")
from src.pipeline.splits import temporal_split
split = temporal_split(df)
X_tr = df.iloc[split.train_idx].reset_index(drop=True)
X_va = df.iloc[split.val_idx].reset_index(drop=True)
X_te = df.iloc[split.test_idx].reset_index(drop=True)
y_tr, y_va, y_te = y[split.train_idx], y[split.val_idx], y[split.test_idx]
print(f"  train={len(X_tr):,}  val={len(X_va):,}  test={len(X_te):,}")


Stage 2/6: temporal split
  train=55,863  val=11,244  test=11,215


In [4]:
print("Stage 3/6: pipeline + model fit")
from sklearn.calibration import CalibratedClassifierCV
from src.models.registry import make_logistic_regression, make_random_forest
from src.pipeline.build import build_pipeline

results = {}
for name, factory in [("LogReg", make_logistic_regression),
                      ("RandomForest", make_random_forest)]:
    pipe = build_pipeline(factory())
    pipe.fit(X_tr, y_tr)
    cal = CalibratedClassifierCV(estimator=pipe, method="isotonic", cv="prefit")
    cal.fit(X_va, y_va)
    results[name] = cal
print(f"  fitted {list(results)}")


Stage 3/6: pipeline + model fit


  fitted ['LogReg', 'RandomForest']


In [5]:
print("Stage 4/6: evaluation")
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from src.eval.calibration import expected_calibration_error
from src.eval.profit_metric import ProfitConfig, total_roi

T_te = X_te["T_eur"].to_numpy()
d_te = X_te["DISTANCE"].to_numpy() * KM_PER_MILE
rows = []
for name, model in results.items():
    p = model.predict_proba(X_te)[:, 1]
    rows.append({
        "model": name,
        "ROC_AUC": roc_auc_score(y_te, p) if y_te.sum() else float("nan"),
        "PR_AUC": average_precision_score(y_te, p),
        "Brier": brier_score_loss(y_te, p),
        "ECE": expected_calibration_error(y_te, p),
        "ROI": total_roi(y_te, p, T_te, d_te, ProfitConfig())["roi"],
    })
table = pd.DataFrame(rows).round(4)
print(table.to_string(index=False))


Stage 4/6: evaluation


       model  ROC_AUC  PR_AUC  Brier    ECE  ROI
      LogReg   0.5561  0.0611 0.0502 0.0057  0.0
RandomForest   0.5206  0.0553 0.0503 0.0051  0.0


In [6]:
print("Stage 5/6: threshold sweep + bankroll policy")
from src.eval.threshold import bankroll_constrained_profit, profit_curve

best_name = table.loc[table["ROI"].idxmax(), "model"]
best_proba = results[best_name].predict_proba(X_te)[:, 1]
curve = profit_curve(y_te, best_proba, T_te, d_te)
top = curve.nlargest(5, "roi")
print(f"  best model: {best_name}")
print("  top 5 global thresholds by ROI:")
print(top[["threshold", "n_buys", "roi", "profit_total_eur"]].to_string(index=False))

bk = bankroll_constrained_profit(y_te, best_proba, T_te, d_te, bankroll_eur=10_000)
print(f"  €10k bankroll → profit=€{bk['profit_total_eur']:,.0f}, ROI={bk['roi']:.2%}, n_buys={bk['n_buys']}")


Stage 5/6: threshold sweep + bankroll policy


  best model: LogReg
  top 5 global thresholds by ROI:
 threshold  n_buys  roi  profit_total_eur
      0.12       0  0.0         -51339.38
      0.13       0  0.0         -51339.38
      0.14       0  0.0         -51339.38
      0.15       0  0.0         -51339.38
      0.16       0  0.0         -51339.38
  €10k bankroll → profit=€-51,339, ROI=0.00%, n_buys=0


In [7]:
print("Stage 6/6: failure-mode summary")
from src.eval.failure_modes import error_table_by

X_te_with_hour = X_te.assign(HOUR=(X_te["CRS_DEP_TIME"].fillna(0).astype(int) // 100))
hour_errors = error_table_by(X_te_with_hour, y_te, best_proba, by="HOUR")
print("Errors by hour (top 12 by volume):")
print(hour_errors.head(12))
print(f"\nTotal pipeline time: {time.time() - t0:.1f}s")


Stage 6/6: failure-mode summary
Errors by hour (top 12 by volume):
      n  pos  tp  fp  fn   tn  base_rate  precision  recall
by                                                         
8   974   52   0   0  52  922   0.053388        0.0     0.0
10  845   41   0   0  41  804   0.048521        0.0     0.0
9   842   48   0   0  48  794   0.057007        0.0     0.0
7   832   57   0   0  57  775   0.068510        0.0     0.0
6   721   41   0   0  41  680   0.056865        0.0     0.0
11  710   29   0   0  29  681   0.040845        0.0     0.0
17  684   56   0   0  56  628   0.081871        0.0     0.0
16  651   21   0   0  21  630   0.032258        0.0     0.0
15  594   27   0   0  27  567   0.045455        0.0     0.0
18  591   27   0   0  27  564   0.045685        0.0     0.0
14  572   23   0   0  23  549   0.040210        0.0     0.0
13  565   24   0   0  24  541   0.042478        0.0     0.0

Total pipeline time: 60.4s


## Summary

This notebook executed:

1. Data load (synthetic fallback if BTS download is unavailable).
2. EC261-aware label generation.
3. Booking-time feature engineering with leakage audit.
4. Train/val/test temporal split.
5. Logistic Regression and Random Forest fits, both isotonic-calibrated.
6. Profit metric, threshold sweep, bankroll policy.
7. Failure-mode summary by hour.

This is the artefact CI runs on every push to guarantee the pipeline does not silently break.